# Geospatial Data Analytics Notebook
This notebook introduces core geospatial workflows using GeoPandas and Shapely, then applies them to New York City neighborhood and population datasets.

## What this notebook covers
- Environment setup and geospatial geometry basics
- GeoDataFrame creation and visualization
- Coordinate reference systems (CRS) and reprojection
- Overlay and attribute enrichment with demographic data
- Buffer-based population analysis and point conversion
- Initial exploration of NYC street tree data

In [ ]:
%pip install geopandas
%pip install matplotlib
%pip install shapely

## 1. Import Libraries
Load the core Python packages used throughout the notebook.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import shapely
import pandas as pd

In [ ]:
from shapely.geometry import Point, LineString, Polygon, MultiPolygon

## 2. Build Basic Geometries
Create simple geometric objects and combine them to understand spatial operations in Shapely.

In [ ]:
# Define the top of the car
top = Polygon([(0, 1), (0, 2), (2, 2), (2, 1)])
top

# Define the bottom part of the car
bottom = Polygon([(-1, 0), (-1, 1), (4, 1), (4, 0)])
bottom

# Create the wheels with a radius of 0.5 each
wheel_front = Point(0, -0.5).buffer(0.5)
wheel_rear = Point(3, -0.5).buffer(0.5)
wheel_rear

# Combine the shapes to form the car
car = top.union(bottom).union(wheel_front).union(wheel_rear)
car

# Calculate the circumference and area of the car


In [ ]:
point = Point(0, 0)
point

circle = point.buffer(1)
circle

In [ ]:
line = LineString([(2,2), (3.5,4), (7,5)])
line

line_buffer = line.buffer(2)
line_buffer

In [ ]:
square = Polygon([(3, 0), (3, 1), (4, 1), (4, 0)])
square

## 3. Create and Enrich a GeoDataFrame
Collect geometries into a GeoDataFrame, derive attributes, and compute basic geometric metrics.

In [ ]:
geodata = {'point':point, 'circle':circle, 'line':line, 'line_buffer':line_buffer, 'square':square}

geodata.keys()

In [ ]:
geodata.values()

In [ ]:
gdf = gpd.GeoDataFrame(geodata.items(), columns = ['geo_name', 'geometry'])
gdf

In [ ]:
gdf['geometry_type'] = gdf['geometry'].apply(lambda x: type(x).__name__)
gdf

In [ ]:
gdf['geometry_buffered'] = gdf['geometry'].apply(lambda x: x.buffer(0.5))
gdf

In [ ]:
gdf['geometry_length'] = [g.length for g in gdf['geometry'].to_list()]
gdf['geometry_area'] = [g.area for g in gdf['geometry'].to_list()]

gdf

In [ ]:
gdf.plot()

## 4. Visualize Synthetic Geometry Data
Use progressively customized maps to understand how styling choices affect interpretation.

In [ ]:
f, ax = plt.subplots(1,1,figsize=(6,6))

gdf.plot(ax=ax)

In [ ]:
f, ax = plt.subplots(1,1,figsize=(6,6))
gdf.plot(ax=ax,
        color = 'crimson',
        edgecolor = 'steelblue',
        linewidth = 2,
        alpha = 0.8)

In [ ]:
f, ax = plt.subplots(1,1,figsize=(6,6))

gdf.plot(ax=ax,
        column = 'geometry_type',
        edgecolor = 'steelblue',
        linewidth = 2,
        alpha = 0.8)

In [ ]:
%pip install geodatasets
from geodatasets import get_path
gdf = gpd.read_file(get_path("nybb"))
gdf

## 5. Load NYC Borough Boundaries
Switch from synthetic examples to real geospatial boundaries from the NYC borough dataset.

In [ ]:
f, ax = plt.subplots(1,1,figsize=(7,7))
gdf.plot(ax=ax, column = 'BoroName', linewidth = 1.5, cmap = 'tab10')

In [ ]:
f, ax = plt.subplots(1,2,figsize=(15,7))

gdf.plot(column = 'Shape_Leng', ax = ax[0], cmap = 'Reds', legend = True)
gdf.plot(column = 'Shape_Area', ax = ax[1], cmap = 'Reds', legend = True)

ax[0].set_title('Shape_Length', fontsize = 20, pad = 10)
ax[1].set_title('Shape_Area', fontsize = 20, pad = 10)

ax[0].axis('off')
ax[1].axis('off')

In [ ]:
nybb = gpd.read_file(get_path("nybb"))
print(nybb.crs)

## 6. Coordinate Reference Systems (CRS)
Inspect CRS metadata and reproject geometries before comparative or global-coordinate analysis.

In [ ]:
nybb.crs

In [ ]:
world_nybb = nybb.to_crs('4326')
print(world_nybb.crs)

In [ ]:
f, ax = plt.subplots(1,1,figsize=(7,7))
world_nybb.plot(ax=ax, edgecolor = 'w', cmap = 'tab20', alpha = 0.5, linewidth = 1)

In [ ]:
file_name = '2010 Neighborhood Tabulation Areas (NTAs).geojson'
gdf_NTA = gpd.read_file(file_name)
print(len(gdf_NTA))
gdf_NTA.head(10)

## 7. Compare Borough and Neighborhood Layers
Load neighborhood polygons, inspect their distribution, and prepare them for spatial comparison with borough boundaries.

In [ ]:
f, ax = plt.subplots(1,2,figsize=(10,5))
gdf_NTA.shape_area.hist(ax=ax[0], bins = 20)
gdf_NTA.shape_leng.hist(ax=ax[1], bins = 20)
ax[0].set_yscale('log')
ax[1].set_yscale('log')

ax[0].set_title('NYC neighborhood areas', fontsize = 14, pad = 16)
ax[1].set_title('NYC neighborhood length', fontsize = 14, pad = 16)

In [ ]:
f, ax = plt.subplots(1,2,figsize=(10,5))

nybb.plot(ax=ax[0], cmap = 'tab20')
gdf_NTA.plot(ax=ax[1], cmap = 'tab20')

ax[0].set_title('NYC boroughs')
ax[1].set_title('NYC neigbhorhoods')

In [ ]:
print(nybb.crs)
print(gdf_NTA.crs)

In [ ]:
gdf_NTA_local = gdf_NTA.to_crs(nybb.crs)
print(gdf_NTA_local.crs)

### Key Analysis: Overlay-Based Neighborhood Counts
This block compares two counting strategies: direct borough grouping versus counts derived from polygon overlays. It also checks agreement using correlation.

In [ ]:
df_group_1 = gdf_NTA_local.groupby(by = 'boro_name').count()[['geometry']].rename(columns = {'geometry' : 'ngh_cnt_1'})
df_group_1

gdf_over = gpd.overlay(nybb, gdf_NTA_local[['ntaname', 'geometry']])
gdf_over.head(3)


df_group_2 = gdf_over.groupby(by = 'BoroName').count()
df_group_2 = df_group_2[['ntaname']].rename(columns = {'ntaname' : 'ngh_cnt_2'})
df_group_2

df_group = df_group_1.merge(df_group_2, left_index = True, right_index = True)
df_group


df_group.corr()

In [ ]:
f, ax = plt.subplots(1,1,figsize=(10,10))
nybb.plot(ax=ax, color = 'none', edgecolor = 'b')
gdf_NTA_local.plot(ax=ax, color = 'none', edgecolor = 'r')

In [ ]:
NY_NTA = pd.read_csv('New_York_City_Population_By_Neighborhood_Tabulation_Areas_20240618.csv')
NY_NTA.head()

gdf_NTA = gpd.read_file('2010 Neighborhood Tabulation Areas (NTAs).geojson')
print(len(gdf_NTA))

## 8. Enrich Neighborhood Geometry with Population Data
Join tabular demographics to neighborhood polygons and create a choropleth map of population.

In [ ]:
print(set(NY_NTA.Year))
NY_NTA = NY_NTA[NY_NTA.Year==2010]

display(NY_NTA.head(1))
display(gdf_NTA.head(1))

In [ ]:
codes_1 = set(NY_NTA['NTA Code'])
codes_2 = set(gdf_NTA['ntacode'])

In [ ]:
len(codes_1), len(codes_2), len(codes_1.intersection(codes_2))

In [ ]:
gdf_merged = gdf_NTA.merge(NY_NTA, right_on = 'NTA Code', left_on = 'ntacode')
# gdf_merged = gdf_merged[['ntacode', 'geometry', 'Population']]
gdf_merged.head(3)

In [ ]:
f, ax = plt.subplots(1,1,figsize=(10,10))

gdf_merged.plot(ax=ax, column = 'Population', cmap = 'Reds')
ax.axis('off')

In [ ]:
gdf_Queens = gdf_merged[gdf_merged.boro_name=='Queens']
print(len(gdf_Queens))

## 9. Buffer Analysis Around Queens
Create neighborhood centroids and estimate how population exposure changes with increasing buffer distance.

In [ ]:
distances = [100, 1000, 5000]
for d in distances:
    gdf_Queens.buffer(d)


In [ ]:
gdf_merged_points = gdf_merged.copy()
gdf_merged_points['geometry'] = [g.centroid for g in list(gdf_merged_points.geometry)]
gdf_merged_points.head(3)

In [ ]:
gdf_merged_points.plot()

In [ ]:
def count_population_stats(distance):
    
    # buffering
    gdf_Queens_buffered = gdf_Queens.copy()
    gdf_Queens_buffered['geometry'] = gdf_Queens_buffered['geometry'].buffer(distance)
    gdf_Queens_buffered.plot()
    
    
    # spatial join
    gdf_joined = gpd.sjoin(gdf_merged_points, gdf_Queens_buffered)
    
    
    # compute population
    print('Zone size: ', distance, ' feet')
    print('Queens population: ', sum(gdf_joined.Population_left))
    print('Buffer zone population: ', sum(gdf_joined.Population_right))
    print('Population ratio: ', round(sum(gdf_joined.Population_left) / sum(gdf_joined.Population_right),2))
    print()

### Key Analysis: Population Ratio by Buffer Distance
The function below buffers Queens neighborhoods, performs a spatial join, and reports the ratio between nearby population and baseline Queens population.

In [ ]:
for distance in distances:
    count_population_stats(distance)

## 10. Geometry Feature Engineering Refresher
Rebuild a small synthetic geometry dataset, compute shape metrics, and style a thematic map using derived geometry attributes.

In [ ]:
top = Polygon([(0,1),(0,2),(2,2),(2,1)])
bottom = Polygon([(-1,0),(-1,1),(4,1),(4,0)])
wheel_front = Point(0, -0.5).buffer(0.5)
wheel_rear = Point(3, -0.5).buffer(0.5)
geometries = [top, bottom, wheel_front, wheel_rear]
gdf = gpd.GeoDataFrame(geometries, columns = ['geometry'])
gdf.plot()
gdf.crs = 4326
display(gdf)

In [ ]:
gdf['geometry_length'] = gdf.length
gdf['geometry_area'] = gdf.area
gdf.head()

In [ ]:
gdf['geometry'] = [g.buffer(0.2) for g in gdf['geometry']]
gdf.head()

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(7, 7))

gdf.plot(column='geometry_length', ax=ax, cmap='Reds', edgecolor='steelblue', linewidth=2, alpha=0.8)


# Turning tabular data into geospatial


In [ ]:
tdf = pd.read_csv('2015_Street_Tree_Census_-_Tree_Data_20240618.csv', nrows = 10000)
len(tdf)
tdf.head(3)

In [ ]:
geometry = [Point(xy) for xy in zip(tdf['longitude'], tdf['latitude'])]
geometry[0]

gdf = gpd.GeoDataFrame(tdf, geometry = geometry)[['tree_id', 'stump_diam', 'status', 'health', 'geometry']]
gdf.crs = 4326
gdf.head()

### Visualize Street Tree Status
Plot sampled street-tree points by status to inspect spatial patterns and data completeness in the tabular-to-geospatial conversion output.

In [ ]:
f, ax = plt.subplots(1,1,figsize=(10,10))

gdf.plot(ax=ax, column = 'status', legend = True, markersize = 10, alpha = 0.4)
ax.axis('off')

## 11. Spatially Join Trees to Neighborhoods
Attach tree points to neighborhood polygons, then prepare the joined records for neighborhood-level aggregation.

In [ ]:
gdf_join = gpd.sjoin(gdf_merged, gdf)
gdf_join.head(3)

In [ ]:
gdf_join = gdf_join[['NTA Name', 'Population', 'tree_id', 'geometry']]
gdf_join.head(3)

### Aggregate Tree Counts and Compute Per-Capita Rates
Count unique trees per neighborhood, merge counts back to polygons, and compute `trees_per_capita` for comparability across neighborhoods.

In [ ]:
gdf_grouped = gdf_join.groupby(by = 'NTA Name').nunique()[['tree_id']]
gdf_grouped = gdf_grouped.rename(columns = {'tree_id' : 'tree_count'})
gdf_grouped

In [ ]:
gdf_join_tree = gdf_merged.merge(gdf_grouped, left_on = 'NTA Name', right_index = True)
gdf_join_tree.head(3)

In [ ]:
gdf_join_tree['trees_per_capita'] = gdf_join_tree['tree_count'] / gdf_join_tree['Population']
gdf_join_tree = gdf_join_tree.sort_values(by = 'trees_per_capita', ascending = False)

In [ ]:
gdf_join_tree[['NTA Name', 'tree_count', 'Population', 'trees_per_capita']].head(10)

In [ ]:
import matplotlib.pyplot as plt

f, ax = plt.subplots(1,1,figsize=(10,10))

gdf_join_tree.plot(ax=ax, column = 'trees_per_capita', cmap = 'Greens', edgecolor = 'k')

ax.axis('off')

### Improve Choropleth Readability with Log Scaling
Apply logarithmic color normalization to better separate neighborhoods with very small and very large `trees_per_capita` values.

In [ ]:
from matplotlib.colors import LogNorm

norm = LogNorm(vmin = gdf_join_tree['trees_per_capita'].min(), 
               vmax = gdf_join_tree['trees_per_capita'].max())


f, ax = plt.subplots(1,1,figsize=(10,10))

gdf_join_tree.plot(ax=ax, 
                   column = 'trees_per_capita', 
                   cmap = 'Greens', 
                   edgecolor = 'k',
                   norm = norm, 
                   legend = True)

ax.axis('off')